In [ ]:
# Install required dependencies
!pip install -q datasets timm torch scikit-learn numpy

import time
import torch
import numpy as np
from sklearn.metrics import classification_report
from datasets import load_dataset
from torch.utils.data import DataLoader
import timm
from timm.data import resolve_model_data_config, create_transform


In [ ]:

# ==========================================
# 1. Configuration & Hyperparameters
# ==========================================
MODEL_NAME = "convnext_tiny"
BATCH_SIZE = 128
LR = 2e-4
WEIGHT_DECAY = 0.2
WARMUP_EPOCHS = 2
EPOCHS = 12
NUM_WORKERS = 2

EMOTION_LABELS = [
    "amusement", "awe", "contentment", "excitement",
    "anger", "disgust", "fear", "sadness"
]
NUM_CLASSES = len(EMOTION_LABELS)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [3]:

# ==========================================
# 2. Data Loading & Preprocessing
# ==========================================
print(f"Resolving native configuration for {MODEL_NAME}...")
data_cfg = resolve_model_data_config(timm.create_model(MODEL_NAME, pretrained=False))
mean, std = data_cfg["mean"], data_cfg["std"]
INPUT_SIZE = data_cfg["input_size"][1] # Native resolution

train_transform = create_transform(
    input_size=INPUT_SIZE, is_training=True, mean=mean, std=std,
    auto_augment="rand-m7-mstd0.5-inc1", re_prob=0.25, re_mode="pixel", re_count=1
)
eval_transform = create_transform(
    input_size=INPUT_SIZE, is_training=False, mean=mean, std=std, crop_pct=0.875
)

print("Downloading EmoSet-118K...")
full_data = load_dataset("Woleek/EmoSet-118K", split="train")

print("Shrinking to a 10% Mini-EmoSet...")
mini_dataset = full_data.train_test_split(train_size=0.10, stratify_by_column="label")['train']

print("Creating balanced Train / Val / Test splits...")
split_1 = mini_dataset.train_test_split(test_size=0.20, stratify_by_column="label")
mini_train = split_1['train']
temp_test = split_1['test']

split_2 = temp_test.train_test_split(test_size=0.50, stratify_by_column="label")
mini_val = split_2['train']
mini_test = split_2['test']

print(f"Train Images: {len(mini_train)} | Val Images: {len(mini_val)} | Test Images: {len(mini_test)}")

def apply_train_transforms(examples):
    examples["pixel_values"] = [train_transform(img.convert("RGB")) for img in examples["image"]]
    return examples

def apply_val_transforms(examples):
    examples["pixel_values"] = [eval_transform(img.convert("RGB")) for img in examples["image"]]
    return examples

mini_train = mini_train.with_transform(apply_train_transforms)
mini_val = mini_val.with_transform(apply_val_transforms)
mini_test = mini_test.with_transform(apply_val_transforms)

def collate_pixels(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    return {"pixel_values": pixel_values, "label": labels}

train_loader = DataLoader(mini_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_pixels)
val_loader = DataLoader(mini_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_pixels)
test_loader = DataLoader(mini_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_pixels)

print(f"DataLoaders ready! (batch_size={BATCH_SIZE}, input_size={INPUT_SIZE} [native])\n")


Resolving native configuration for convnext_tiny...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

...

Generating train split:   0%|          | 0/94481 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/5905 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/17716 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/18 [00:00<?, ?it/s]

Shrinking to a 10% Mini-EmoSet...
Creating balanced Train / Val / Test splits...
Train Images: 7558 | Val Images: 945 | Test Images: 945
DataLoaders ready! (batch_size=128, input_size=224 [native])



In [5]:

# ==========================================
# 3. Model Initialization
# ==========================================
print(f"=== Initializing {MODEL_NAME} ===")
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES)

# Freeze backbone, leave classifier trainable for Phase 1
for param in model.parameters():
    param.requires_grad = False
for param in model.get_classifier().parameters():
    param.requires_grad = True

frozen_count = sum(1 for p in model.parameters() if not p.requires_grad)
trainable_count = sum(1 for p in model.parameters() if p.requires_grad)
print(f"Frozen params: {frozen_count}, Trainable params: {trainable_count}")

model = model.to(device)

# Calculate class weights for imbalanced datasets
labels_list = [mini_train[i]['label'] for i in range(len(mini_train))]
counts = np.bincount(labels_list, minlength=NUM_CLASSES)
class_weights = torch.tensor(len(labels_list) / (NUM_CLASSES * counts), dtype=torch.float32).to(device)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

# Attempt to compile the model for faster training on modern GPUs
print("Attempting to compile model with torch.compile...")
try:
    active_model = torch.compile(model, mode="default")
    print("torch.compile enabled.")
except Exception:
    active_model = model
    print("torch.compile unavailable or failed, proceeding natively.")


=== Initializing convnext_tiny ===
Frozen params: 180, Trainable params: 2
Attempting to compile model with torch.compile...
torch.compile enabled.


In [6]:

# ==========================================
# 4. Training Loop (Phase 1 & Phase 2)
# ==========================================
optimizer = torch.optim.AdamW([p for p in active_model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(WARMUP_EPOCHS, 1))
scaler = torch.amp.GradScaler()

train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_val_loss = float("inf")
best_state = None
total_start = time.time()

for epoch in range(EPOCHS):
    if epoch == 0 and WARMUP_EPOCHS > 0:
        print("\n--- Phase 1: Warmup (frozen backbone, training head only) ---")

    if epoch == WARMUP_EPOCHS and WARMUP_EPOCHS < EPOCHS:
        print("\n--- Phase 2: Fine-tuning (unfreezing backbone) ---")
        actual_model = active_model._orig_mod if hasattr(active_model, "_orig_mod") else active_model

        # Unfreeze all parameters
        for param in actual_model.parameters():
            param.requires_grad = True

        # Differential learning rates
        classifier_param_ids = {id(p) for p in actual_model.get_classifier().parameters()}
        optimizer = torch.optim.AdamW([
            {"params": [p for p in actual_model.parameters() if id(p) in classifier_param_ids], "lr": LR},
            {"params": [p for p in actual_model.parameters() if id(p) not in classifier_param_ids], "lr": LR * 0.1},
        ], weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS)

    phase = "Phase 1" if epoch < WARMUP_EPOCHS else "Phase 2"
    epoch_start = time.time()

    # Train
    active_model.train()
    running_loss, correct, total = 0.0, 0, 0
    for batch in train_loader:
        imgs = batch["pixel_values"].to(device)
        lbls = batch["label"].to(device)
        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = active_model(imgs)
            loss = criterion(outputs, lbls)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(active_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == lbls).sum().item()
        total += lbls.size(0)

    scheduler.step()
    train_losses.append(running_loss / total)
    train_accs.append(correct / total)

    # Validate
    active_model.eval()
    vloss, vcorrect, vtotal = 0.0, 0, 0
    with torch.no_grad():
        for batch in val_loader:
            imgs = batch["pixel_values"].to(device)
            lbls = batch["label"].to(device)
            with torch.amp.autocast("cuda"):
                outputs = active_model(imgs)
                loss = criterion(outputs, lbls)
            vloss += loss.item() * imgs.size(0)
            vcorrect += (outputs.argmax(1) == lbls).sum().item()
            vtotal += lbls.size(0)

    val_losses.append(vloss / vtotal)
    val_accs.append(vcorrect / vtotal)

    epoch_time = time.time() - epoch_start
    print(f"[{phase}] Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_losses[-1]:.4f} Acc: {train_accs[-1]:.4f} | Val Loss: {val_losses[-1]:.4f} Acc: {val_accs[-1]:.4f} | Time: {epoch_time:.1f}s")

    # Save best model
    if val_losses[-1] < best_val_loss:
        best_val_loss = val_losses[-1]
        src = active_model._orig_mod if hasattr(active_model, "_orig_mod") else active_model
        best_state = {k: v.cpu().clone() for k, v in src.state_dict().items()}

# Load best weights
src = active_model._orig_mod if hasattr(active_model, "_orig_mod") else active_model
src.load_state_dict(best_state)
print(f"\nTraining complete. Best val loss: {best_val_loss:.4f} | Total time: {time.time() - total_start:.1f}s")



--- Phase 1: Warmup (frozen backbone, training head only) ---


W0621 21:27:43.407000 1138 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


[Phase 1] Epoch 1/12 | Train Loss: 1.8445 Acc: 0.3285 | Val Loss: 1.4273 Acc: 0.5090 | Time: 165.6s
[Phase 1] Epoch 2/12 | Train Loss: 1.3597 Acc: 0.5322 | Val Loss: 1.2312 Acc: 0.5894 | Time: 67.9s

--- Phase 2: Fine-tuning (unfreezing backbone) ---
[Phase 2] Epoch 3/12 | Train Loss: 1.1808 Acc: 0.5942 | Val Loss: 1.0433 Acc: 0.6434 | Time: 68.8s
[Phase 2] Epoch 4/12 | Train Loss: 1.0382 Acc: 0.6413 | Val Loss: 0.9623 Acc: 0.6656 | Time: 68.5s
[Phase 2] Epoch 5/12 | Train Loss: 0.9842 Acc: 0.6544 | Val Loss: 0.9158 Acc: 0.6804 | Time: 68.9s
[Phase 2] Epoch 6/12 | Train Loss: 0.9303 Acc: 0.6625 | Val Loss: 0.8904 Acc: 0.6910 | Time: 68.5s
[Phase 2] Epoch 7/12 | Train Loss: 0.9152 Acc: 0.6772 | Val Loss: 0.8735 Acc: 0.6963 | Time: 69.6s
[Phase 2] Epoch 8/12 | Train Loss: 0.8965 Acc: 0.6764 | Val Loss: 0.8613 Acc: 0.7026 | Time: 68.3s
[Phase 2] Epoch 9/12 | Train Loss: 0.8851 Acc: 0.6862 | Val Loss: 0.8538 Acc: 0.7026 | Time: 68.5s
[Phase 2] Epoch 10/12 | Train Loss: 0.8783 Acc: 0.6801 |

In [7]:

# ==========================================
# 5. Evaluation
# ==========================================
print(f"\n=== Test Set Evaluation: {MODEL_NAME} ===")
eval_model = active_model._orig_mod if hasattr(active_model, "_orig_mod") else active_model
eval_model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        imgs = batch["pixel_values"].to(device)
        lbls = batch["label"]
        with torch.amp.autocast("cuda"):
            outputs = eval_model(imgs)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(lbls.numpy())

print(classification_report(np.array(all_labels), np.array(all_preds), target_names=EMOTION_LABELS))



=== Test Set Evaluation: convnext_tiny ===
              precision    recall  f1-score   support

   amusement       0.71      0.64      0.67       157
         awe       0.75      0.80      0.77       120
 contentment       0.54      0.57      0.55       130
  excitement       0.75      0.69      0.72       158
       anger       0.67      0.73      0.70        85
     disgust       0.76      0.88      0.82        85
        fear       0.71      0.66      0.68       108
     sadness       0.63      0.63      0.63       102

    accuracy                           0.69       945
   macro avg       0.69      0.70      0.69       945
weighted avg       0.69      0.69      0.69       945

